In [ ]:
import math
import time
import torchvision
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import nn, Tensor
from sklearn.datasets import make_moons
from sklearn.datasets import make_swiss_roll
from tqdm import tqdm

# flow_matching
from flow_matching.path import AffineProbPath, CondOTProbPath
from flow_matching.path.scheduler import (
    CondOTScheduler, PolynomialConvexScheduler, LinearVPScheduler, CosineScheduler
)
from flow_matching.solver import ODESolver 
from flow_matching.utils import ModelWrapper 

# visualization
import matplotlib.pyplot as plt
from matplotlib import cm


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

#### Dataset

In [ ]:
BATCH_SIZE = 128

data = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('./data',
    transform=torchvision.transforms.Compose([
        torchvision.transforms.ToTensor()
    ]),
    download=True),
    batch_size=BATCH_SIZE,
    shuffle=True)

x, y = next(iter(data))     # Shape: (batch_size, 1, 28, 28)
x_sample = torch.permute(x[0], (1, 2, 0))

ax = plt.subplot(211)
ax.imshow(x_sample, cmap="gray")
plt.show()

print(f"x_shape: {x.shape}")
print(f"min: {torch.min(x_sample)} max: {torch.max(x_sample)}")
print(f"mean: {torch.mean(x)}, var: {torch.var(x)}")

In [ ]:
print(x.shape)
print(x.view(x.shape[0], -1).shape)

print(torch.randn_like(x.view(x.shape[0], -1)).shape)

#### Model Creation

In [ ]:
# -- Essential Convolution blocks
class Downsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels,
                              kernel_size=3,
                              stride=2,
                              padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels=in_channels,
                                       out_channels=out_channels,
                                       kernel_size=4,
                                       stride=2,
                                       padding=1)
    
    def forward(self, x):
        return self.conv(x)


class Conv2dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, residual=False, n_groups=1):
        super().__init__()
        self.residual = residual

        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, 
                      kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=n_groups, num_channels=out_channels),
            nn.Mish()
        )
    
    def forward(self, x):
        if self.residual:
            return F.mish(x + self.block(x))
        else:
            return self.block(x)


class DoubleConv(nn.Module):
    """
    Double Convolution block with slight activation func deviation
    (convolution => [Norm] => Mish) * 2
    """
    def __init__(self, in_channels, out_channels, mid_channels=None, n_groups=1):
        super().__init__()

        # Determine convolutional channels for mid
        if not mid_channels:
            mid_channels = out_channels

        # Double convolution block
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=mid_channels, 
                      kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=n_groups, num_channels=mid_channels),
            nn.Mish(),      # Deviation from Relu
            nn.Conv2d(in_channels=mid_channels, out_channels=out_channels, 
                      kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=n_groups, num_channels=out_channels),
            nn.Mish(),      # Deviation from Relu
        )

    def forward(self, x):
        return self.block(x)


class DownMaxPoolConv(nn.Module):
    """
    Downscaling with maxpool then double conv
    """
    def __init__(self, in_channels, out_channels, mid_channels=None, n_groups=1,
                 time_emb_dim=256):
        super().__init__()

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(kernel_size=2),
            DoubleConv(in_channels=in_channels, out_channels=out_channels,
                       mid_channels=mid_channels, n_groups=n_groups)
        )

        # Time-embedding layer
        self.time_emb_layer = nn.Sequential(
            nn.Mish(),
            nn.Linear(
                time_emb_dim,
                out_channels
            ),
        )
    
    def forward(self, x, t=None):
        if t is not None:
            x = self.maxpool_conv(x)
            # Broadcast time embedding to (B, C, H, W)
            # t = t[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
            t = self.time_emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
            return x + t
        else:
            return self.maxpool_conv(x)


class UpConv(nn.Module):
    """
    Upscaling with conv transpose then double conv
    """
    def __init__(self, in_channels, out_channels, mid_channels=None, n_groups=1,
                 time_emb_dim=256):
        super().__init__()
        # Upsample with conv transpose but have number of channels halve
        self.up_conv = nn.Sequential(
            nn.ConvTranspose2d(in_channels=in_channels, out_channels=in_channels // 2,
                               kernel_size=2, stride=2),
            DoubleConv(in_channels=in_channels // 2, out_channels=out_channels,
                       mid_channels=mid_channels, n_groups=n_groups)
        )

        # Individual Up Sample blocks + Double conv
        self.up = nn.ConvTranspose2d(in_channels=in_channels, out_channels=in_channels // 2,
                                     kernel_size=4, stride=2, padding=1)
        self.double_conv = DoubleConv(in_channels=in_channels, out_channels=out_channels,
                                      mid_channels=mid_channels, n_groups=n_groups)
        
        # Time-embedding layer
        self.time_emb_layer = nn.Sequential(
            nn.Mish(),
            nn.Linear(
                time_emb_dim,
                out_channels
            ),
        )
    
    def forward(self, x1, x2=None, t=None):
        if x2 is not None:
            # -- Deal with residual
            # Input is of (Batch size, channels, height, width)
            # TODO: Deal with padding

            # Upscale first
            x1 = self.up(x1)

            # Concat along channels axis
            x = torch.cat([x2, x1], dim=1)

            x = self.double_conv(x)
        else:
            # Otherwise up scale and double conv
            x = self.up_conv(x1)

        # Deal with time-embedding
        if t is not None:
            # t = t[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
            t = self.time_emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
            return x + t
        else:
            return x
        

class SelfAttention(nn.Module):
    def __init__(self, channels, size):
        super(SelfAttention, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        # End x: (-1, self.size*self.size, self.channels)
        # Usually its x: (seq_len, batch_size, input_dim)
        # query inputs are: (L, N, Eq) --> (seq_len, batch_size, emb_dim)
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        # self.mha(query, key, value)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)

# -- Position embeddings
class SinusoidalPosEmb(nn.Module):
    # TODO: Investigate
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

# -- Flow model
class SimpleFlowModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, time_dim: int = 1):
        super().__init__()
        self.time_dim = time_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        if t.dim() == 0:
            t = t.expand(x.shape[0], self.time_dim)
        else:
            t = t.view(-1, self.time_dim)

        # t = t.view(-1, 1)
        x = x.view(x.shape[0], -1)

        out = self.net(torch.cat((t, x), dim=-1))
        return out.view(x.shape[0], 1, 28, 28)      # TODO: hardcoded for MNIST, need to generalize for other datasets
    
class UNetFlowModel(nn.Module):
    """
    U-Net with Attention 
    Based on architecture specified in https://arxiv.org/pdf/1505.04597 (fig1)
    Implementation details: https://github.com/milesial/Pytorch-UNet/blob/master/unet/unet_model.py
    """
    def __init__(self, c_in=1, c_out=1, time_dim=256, device="cuda"):
        super().__init__()

        self.device = device
        self.time_dim = time_dim    # Time diffusion step embedding dimension

        # -- Diffusion step encoder
        # Time embeddings
        diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(dim=self.time_dim),
            # nn.Linear(self.time_dim, self.time_dim*4),
            # nn.Mish(),
            # nn.Linear(self.time_dim*4, self.time_dim),
        )
        self.diffusion_step_encoder = diffusion_step_encoder

        # -- U-net blocks
        self.inc = DoubleConv(in_channels=c_in, out_channels=64)
        self.down1 = DownMaxPoolConv(in_channels=64, out_channels=128)      # (H,W) = (14,14)
        self.sa1 = SelfAttention(channels=128, size=14)
        self.down2 = DownMaxPoolConv(in_channels=128, out_channels=256)     # (H,W) = (7,7)
        
        # self.bot = DoubleConv(in_channels=256, out_channels=256)

        self.up1 = UpConv(in_channels=256, out_channels=128)    # (7,7) --> (14,14)
        self.sa2 = SelfAttention(channels=128, size=14)
        self.up2 = UpConv(in_channels=128, out_channels=64)
        self.outc = nn.Conv2d(in_channels=64, out_channels=c_out, 
                              kernel_size=3, padding=1)

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        # -- Time Embeddings
        # Get timestep embeddings
        timesteps = t
        if not torch.is_tensor(timesteps):
            # TODO: this requires sync between CPU and GPU. So try to pass timesteps as tensors if you can
            timesteps = torch.tensor([timesteps], dtype=torch.long, device=x.device)
        elif torch.is_tensor(timesteps) and len(timesteps.shape) == 0:
            timesteps = timesteps[None].to(x.device)
        # broadcast to batch dimension in a way that's compatible with ONNX/Core ML
        timesteps = timesteps.expand(x.shape[0])
        t_emb = self.diffusion_step_encoder(timesteps)

        # -- Sample Forward pass
        x1 = self.inc(x)
        x2 = self.down1(x1, t_emb)
        x2 = self.sa1(x2)
        x3 = self.down2(x2, t_emb)      # Bot already covered in down2

        x = self.up1(x3, x2, t_emb)
        x = self.sa2(x)
        x = self.up2(x, x1, t_emb)
        logits = self.outc(x)
        return logits


In [ ]:
x_test = torch.randn((BATCH_SIZE, 1, 28, 28)).to(device)
model = UNetFlowModel().to(device)
out = model(x=x_test, t=0.5)

print("Output shape:", out.shape)

#### Training

In [ ]:
# -- Params
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EPOCHS = 50

# -- Dataset 
train_dataloader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('./data',
    transform=torchvision.transforms.ToTensor(),
    download=True),
    batch_size=BATCH_SIZE,
    shuffle=True)

# -- Define paths
# affine path with alpha_t = t, sigma_t = 1 - t
path = AffineProbPath(scheduler=CondOTScheduler())

# -- Model
# model = SimpleFlowModel(input_dim=28*28, hidden_dim=512, time_dim=1).to(device)
model = UNetFlowModel().to(device)
model.train()

# -- Training
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()
losses = []

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    epoch_loss = 0.0
    pbar = tqdm(train_dataloader)

    for i, (images, labels) in enumerate(pbar):
        # Randomize time t ~ Uniform(0, 1)
        t = torch.rand(images.size(0)).to(device)

        # Sample x1 and x0 (x0 is gaussian)
        x1 = images.to(device)
        _ = labels.to(device)

        x0 = torch.randn_like(x1).to(device)

        # Sample path
        sample = path.sample(t=t, x_0=x0, x_1=x1)

        # Compute loss and optimize
        loss = torch.pow(model(t=t, x=sample.x_t) - sample.dx_t, 2).mean()

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        # Update epoch loss and progress bar
        epoch_loss += loss.item()
        pbar.set_description(f"Loss: {epoch_loss / (i + 1):.4f}")
    
    losses.append(epoch_loss / len(train_dataloader))

In [ ]:
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Epoch Loss")
plt.title("Training Loss")
plt.show()

#### Sampling

In [ ]:
# Convert from denoiser to velocity prediction
class VelocityModel(ModelWrapper):
    def __init__(self, denoiser: nn.Module, path: AffineProbPath):
        super().__init__(denoiser)
        self.path = path

    def forward(self, t: Tensor, x: Tensor, **extras) -> Tensor:
        x_1_predictions = super().forward(t=t, x=x, **extras)
        return self.path.target_to_velocity(x_1=x_1_predictions, x_t=x, t=t)

In [ ]:
num_steps = 10
batch_size = 256
T = torch.linspace(0, 1, num_steps) 

velocity_model = VelocityModel(denoiser=model, path=path)
solver = ODESolver(velocity_model=velocity_model)

x_init = torch.randn((BATCH_SIZE, 1, 28, 28), dtype=torch.float32, device=device)
x_1 = solver.sample(
    time_grid=T.to(device),
    x_init=x_init,
    method='midpoint',
    step_size=1.0 / num_steps,
    return_intermediates=True,
)

print("Sampled x_1 shape:", x_1.shape)
x_1 = x_1.permute(0, 1, 3, 4, 2)
print("Permuted x_1 shape:", x_1.shape)

x_1 = x_1.cpu().numpy()
print("Numpy x_1 shape:", x_1.shape)

In [ ]:
# -- Visualization of samples
import numpy as np
x_min, x_max = x_1.min(), x_1.max()
_, axs = plt.subplots(1, num_steps, figsize=(20, 3.2))

for i in range(num_steps):
    rand_idx = np.random.randint(0, x_1.shape[1], (1,)).item()
    # axs[i].scatter(x_1[i, :, 0], x_1[i, :, 1], s=5, alpha=0.5)
    axs[i].imshow(x_1[i, rand_idx, :, :], cmap="gray")
    axs[i].set_title(f"t={T[i].item():.2f}")
    # axs[i].set_xlim(x_min, x_max)
    # axs[i].set_ylim(x_min, x_max)
    axs[i].set_aspect('equal')